# AnomalyDetector
`capabilities/analysis/anomaly_detector.py`

Detects unusual regions in plant images **without any labelled training data**, using DINOv2 visual features.

**How it works:**
1. Extract deep visual features from the image using DINOv2 (a powerful self-supervised vision transformer)
2. Compare features to a reference distribution of "normal" images
3. Patches far from normal = anomalous

| Operating mode | When to use |
|---------------|-------------|
| **Reference mode** | You have 5+ healthy images → fit a reference, score new images against it |
| **Self mode** | No reference yet → detect internal patch inconsistencies within one image |

| Model | Size | Speed (CPU) |
|-------|------|-------------|
| `facebook/dinov2-small` | 90 MB | 2–5s (default) |
| `facebook/dinov2-base`  | 330 MB | 5–10s |
| `facebook/dinov2-large` | 1.1 GB | 15–30s |

## Setup

In [ ]:
import sys, os, tempfile, json
sys.path.insert(0, os.path.abspath('../..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from capabilities.analysis.anomaly_detector import AnomalyDetector
from sprout_detection.utils.image_gen import make_sprout_image, make_bare_soil_image

tmp = tempfile.mkdtemp()

# Create test images
healthy_paths = []
for i in range(5):
    p = os.path.join(tmp, f'healthy_{i}.png')
    cv2.imwrite(p, make_sprout_image(seed=i))
    healthy_paths.append(p)

# Create a diseased image with spots
def make_diseased(seed, n_spots=45):
    rng = np.random.default_rng(seed)
    img = make_sprout_image(seed=seed)
    h, w = img.shape[:2]
    for _ in range(n_spots):
        cx = int(rng.integers(10, w-10))
        cy = int(rng.integers(10, h-10))
        r  = int(rng.integers(4, 14))
        color = (int(rng.integers(120, 180)), int(rng.integers(30, 80)), int(rng.integers(30, 80)))
        cv2.circle(img, (cx, cy), r, color, -1)
    return img

diseased_path = os.path.join(tmp, 'diseased.png')
bare_path     = os.path.join(tmp, 'bare.png')
cv2.imwrite(diseased_path, make_diseased(seed=99))
cv2.imwrite(bare_path,     make_bare_soil_image(seed=99))

print(f'{len(healthy_paths)} healthy images + 1 diseased + 1 bare soil')

## Mode 1 — Self-mode (no reference)

In [ ]:
detector = AnomalyDetector(anomaly_threshold=0.50)

# Self-mode: no fit_reference() called
result_healthy  = detector.detect(healthy_paths[0])
result_diseased = detector.detect(diseased_path)
result_bare     = detector.detect(bare_path)

print('Self-mode anomaly scores:')
print(f'  Healthy  : {result_healthy.anomaly_score:.3f}  anomalous={result_healthy.is_anomalous}')
print(f'  Diseased : {result_diseased.anomaly_score:.3f}  anomalous={result_diseased.is_anomalous}')
print(f'  Bare soil: {result_bare.anomaly_score:.3f}  anomalous={result_bare.is_anomalous}')
print()
print(f'Reference fitted: {detector.reference_fitted}')
print(f'Inference time  : {result_diseased.duration_ms:.0f} ms')

## Mode 2 — Reference mode (fit on healthy images)

In [ ]:
# Fit reference distribution from healthy images
detector.fit_reference(healthy_paths)

print(f'Reference fitted: {detector.reference_fitted}')

# Now score images against the reference
ref_healthy  = detector.detect(healthy_paths[0])
ref_diseased = detector.detect(diseased_path)
ref_bare     = detector.detect(bare_path)

print('\nReference-mode anomaly scores:')
print(f'  Healthy  : {ref_healthy.anomaly_score:.3f}  anomalous={ref_healthy.is_anomalous}')
print(f'  Diseased : {ref_diseased.anomaly_score:.3f}  anomalous={ref_diseased.is_anomalous}')
print(f'  Bare soil: {ref_bare.anomaly_score:.3f}  anomalous={ref_bare.is_anomalous}')

## Anomaly score comparison — self vs reference mode

In [ ]:
images = ['Healthy', 'Diseased', 'Bare soil']
self_scores = [result_healthy.anomaly_score, result_diseased.anomaly_score, result_bare.anomaly_score]
ref_scores  = [ref_healthy.anomaly_score,    ref_diseased.anomaly_score,    ref_bare.anomaly_score]

x = np.arange(len(images))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - w/2, self_scores, w, label='Self mode',      color='#3498db', alpha=0.85)
bars2 = ax.bar(x + w/2, ref_scores,  w, label='Reference mode', color='#e74c3c', alpha=0.85)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Threshold (0.50)')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(images, fontsize=10)
ax.set_ylabel('Anomaly Score')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
ax.set_title('Anomaly Scores — Self Mode vs Reference Mode', fontsize=11)
plt.tight_layout()
plt.show()

## Spatial anomaly map

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))

for col, (name, path, r) in enumerate([
    ('Healthy',   healthy_paths[0], ref_healthy),
    ('Diseased',  diseased_path,    ref_diseased),
    ('Bare soil', bare_path,        ref_bare),
]):
    # Row 0: original image
    axes[0, col].imshow(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(f'{name}\nscore={r.anomaly_score:.3f}  anomalous={r.is_anomalous}', fontsize=9)
    axes[0, col].axis('off')
    
    # Row 1: anomaly heatmap
    if r.anomaly_map is not None:
        im = axes[1, col].imshow(r.anomaly_map, cmap='jet')
        plt.colorbar(im, ax=axes[1, col])
        axes[1, col].set_title('Anomaly Heatmap\n(bright = more anomalous)', fontsize=8)
    else:
        axes[1, col].text(0.5, 0.5, 'No heatmap\n(run with real model)',
                          transform=axes[1, col].transAxes, ha='center', va='center', color='gray')
    axes[1, col].axis('off')

plt.suptitle('AnomalyDetector — Spatial Heatmaps (Reference Mode)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Batch scan — screen many images at once

In [ ]:
# Mix healthy and diseased images in a batch
diseased_paths = []
for i in range(3):
    p = os.path.join(tmp, f'diseased_{i}.png')
    cv2.imwrite(p, make_diseased(seed=100+i, n_spots=30+i*10))
    diseased_paths.append(p)

all_images = healthy_paths[:3] + diseased_paths
labels     = ['healthy']*3 + ['diseased']*3

results = [detector.detect(p) for p in all_images]

print(f'Batch scan — {len(results)} images (threshold={detector._threshold}):\n')
print(f'{"Image":<25} {"True label":<12} {"Score":<8} {"Flagged?"}')
print('-' * 55)
for path, true_label, r in zip(all_images, labels, results):
    flagged = '⚠️  YES' if r.is_anomalous else '✅  no'
    correct = '' if (r.is_anomalous == (true_label == 'diseased')) else '  ← miss'
    print(f'{os.path.basename(path):<25} {true_label:<12} {r.anomaly_score:<8.3f} {flagged}{correct}')

## JSON serialisation

In [ ]:
print(ref_diseased.to_json())

## Tips for best results

```
✅ DO:
  - Use 10+ reference images for better distribution coverage
  - Reference images should match your deployment camera/lighting
  - Combine with HSVSegmentor mask to exclude background noise
  - Use reference mode whenever possible — far more accurate

⚠️  TUNE:
  - anomaly_threshold: lower = more sensitive (more false positives)
  - Model size: dinov2-base is significantly better than dinov2-small
    for subtle disease signs

🚫 AVOID:
  - Using only 1–2 reference images (noisy distribution)
  - Mixing different camera angles in the reference set
```